In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [15]:

import os
import random
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


In [16]:
# ✅ Dataset
class SiameseCatsDogsDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        
        cat_dir = os.path.join(root_dir, "cats")
        dog_dir = os.path.join(root_dir, "dogs")

        self.cat_images = [os.path.join(cat_dir, f) for f in os.listdir(cat_dir)
                           if os.path.isfile(os.path.join(cat_dir, f)) and f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.dog_images = [os.path.join(dog_dir, f) for f in os.listdir(dog_dir)
                           if os.path.isfile(os.path.join(dog_dir, f)) and f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        self.all_images = self.cat_images + self.dog_images

    def __getitem__(self, idx):
        img1_path = self.all_images[idx]
        label1 = 0 if "cats" in img1_path else 1
        img1 = Image.open(img1_path).convert("L")

        should_match = random.randint(0, 1)
        if should_match:
            img2_path = random.choice(self.cat_images if label1 == 0 else self.dog_images)
        else:
            img2_path = random.choice(self.dog_images if label1 == 0 else self.cat_images)

        label = torch.tensor([int("cats" in img1_path) == int("cats" in img2_path)], dtype=torch.float32)
        img2 = Image.open(img2_path).convert("L")

        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)

        return img1, img2, label

    def __len__(self):
        return len(self.all_images)

In [17]:
# ✅ Transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [18]:
# ✅ Dataset and Dataloader
data_dir = "/kaggle/input/cats-and-dogs-image-classification/test"  # Adjust as needed
dataset = SiameseCatsDogsDataset(data_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [23]:
# ✅ Siamese Network
class SiameseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=5), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 128, kernel_size=5), nn.ReLU(), nn.MaxPool2d(2)
        )
        # Dynamically calculate the output size of CNN
        self._to_linear = None
        self._get_flattened_size()

        self.fc = nn.Sequential(
            nn.Linear(self._to_linear, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )

    def _get_flattened_size(self):
        with torch.no_grad():
            x = torch.zeros(1, 1, 224, 224)
            x = self.cnn(x)
            self._to_linear = x.view(1, -1).shape[1]

    def forward_once(self, x):
        x = self.cnn(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

    def forward(self, x1, x2):
        emb1 = self.forward_once(x1)
        emb2 = self.forward_once(x2)
        return F.pairwise_distance(emb1, emb2)

In [24]:
# ✅ Contrastive Loss
def contrastive_loss(dist, label, margin=1.0):
    return (1 - label) * dist**2 + label * torch.clamp(margin - dist, min=0)**2

In [25]:
# ✅ Training
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SiameseNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [27]:
model.train()
for epoch in range(10):  # Increase if needed
    running_loss = 0.0
    for img1, img2, label in tqdm(dataloader):
        img1, img2, label = img1.to(device), img2.to(device), label.to(device)
        dist = model(img1, img2)
        loss = contrastive_loss(dist, label).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch [{epoch+1}] Loss: {running_loss / len(dataloader):.4f}")


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch [1] Loss: 0.0062


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch [2] Loss: 0.0062


  0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch [8] Loss: 0.0125


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch [9] Loss: 0.0062


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch [10] Loss: 0.0062
